## Setup  

for example defined in section 4.4 from [https://arxiv.org/pdf/2606.05487](https://arxiv.org/pdf/2606.05487).

In [ ]:
import functools
import jax.numpy as jnp
import jax
from jax.typing import ArrayLike
import matplotlib.pyplot as plt
import numpy as np

import src.geometry as _geom
import src.mesher as _mesher
import src.material as _mat
import src.utils as _utils
import src.bc as _bc
import src.fe_fluid as _fea
import src.fe_transient_transport as _fea_transient_transport
import src.ct_radon_transform as _radon


import src.solver as _solv
import src.mma as _mma
import src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

_Ext = _utils.Extent

_FluidField = _fea.FluidField
_ConcentrationField = _fea_transient_transport.ConcentrationField
_cmap = _viz.fluid_cmap

In [ ]:
# To fix the latex error.
import matplotlib
matplotlib.rcParams['text.usetex'] = False

# Define Geometry and Mesh 

Below we build the **artery** benchmark commonly used in fluid TO.

#### Geometry  
A simple artery is described in a json and passed as the geometry. This geometry is passed to the mesher.

#### Discretisation  
* The domain $(\Omega)$ is partitioned into **bilinear quadrilateral (Q1) elements**.  
* Both primary fields—velocity $(\mathbf u=(u_x,u_y))$ **and** pressure (p\)—are interpolated with the same Q1 shape functions (equal-order Q1/Q1 formulation).

#### Numerical integration  
Element contributions (mass, convection, brinkman, viscous and stability terms) are evaluated with a $(2 \times 2)$ Gauss quadrature.

This mesh serves as the baseline for the finite-element Navier–Stokes solve and for any subsequent density-based topology-optimisation runs.

In [ ]:
#load the sinogram ground truth data
sngrm_grndtruth = jnp.load("sngrm_grndtruth_topopt_geom_vel_stenosis.npy")


In [ ]:
geom = _geom.BrepGeometry(brep_file="./brep/artery_no_ext_stenosis.json")

# nelx_desired, nely_desired, gauss_order = 98, 70, 2
nelx_desired, nely_desired, gauss_order = 80, 80, 2


_fluid_mesh = _mesher.grid_mesh_brep(
  brep=geom,
  nelx_desired=nelx_desired,
  nely_desired=nely_desired,
  dofs_per_node=3,
  gauss_order=gauss_order,
)

density = jnp.zeros(_fluid_mesh.num_elems)
_viz.plot_grid_mesh(_fluid_mesh, density)
_viz.plot_brep(geom)

In [ ]:
mat_params = _mat.FluidMaterial(
  mass_density=1.058,
  dynamic_viscosity=3.45e-2, 
)

## Other fluid material properties

In [ ]:
dens_filter = _utils.create_density_filter(
  _fluid_mesh.elem_centers,
  cutoff_distance=0.03 * _fluid_mesh.bounding_box.diag_length,
  filter_type=_utils.Filters.CIRCULAR,
)

# Boundray Conditions



| Region | Type | Imposed values |
|--------|------|----------------|
| **Inlet top and bottom(left vertical edge)** | Dirichlet | $(u = parabolic)$, $(v = 0)$ |
| **Outlet bottom  (right vertical edge)** | Dirichlet v, p | \(p = 0\), \(v = 0\) |
| **Top wall** | No-slip (Dirichlet) | \(u = 0\), \(v = 0\) |
| **Bottom wall** | No-slip (Dirichlet) | \(u = 0\), \(v = 0\) |

**Characteristic velocity**

The inlet velocity is chosen to give a Reynolds number of 7.68:

$$
U_c \;=\; \frac{\mathrm{Re}\,\nu}{H},
\qquad
\text{Re} = 7.68,\;
\quad
\nu = \frac{\mu}{\rho},
\quad
H = \text{characteristic length}.
$$


In [ ]:
inlet_fraction = 1.0 / 3.0
char_length = inlet_fraction * _fluid_mesh.bounding_box.ly
reynolds_num = 7.68
char_velocity = reynolds_num * mat_params.kinematic_viscosity / (char_length)

In [ ]:
face_tol = _fluid_mesh.elem_size[0] * 0.5

# inlet top condition
inlet_top_faces = _bc.identify_faces(_fluid_mesh, edges=[geom.edges[1]], tol=face_tol)
n = len(inlet_top_faces)
x_nodes = jnp.linspace(-1.0, 1.0, n + 1)
u_profile = char_velocity * (1.0 - x_nodes**2)
face_node_vals = jnp.stack([u_profile[1:], u_profile[:-1]], axis=1)
u_vel = (_FluidField.U_VEL, face_node_vals)
v_vel = (_FluidField.V_VEL, jnp.zeros_like(face_node_vals))
inlet_top_face_val = [u_vel, v_vel]

# wall condition
wall_edge_nos = [0, 2, 3, 4, 6, 7]
wall_edges = [geom.edges[i] for i in wall_edge_nos]
wall_faces = _bc.identify_faces(_fluid_mesh, edges=wall_edges, tol=face_tol)
n = len(wall_faces)
u_vel = (_FluidField.U_VEL, jnp.zeros(n))
v_vel = (_FluidField.V_VEL, jnp.zeros(n))
wall_face_val = [u_vel, v_vel]


# outlet condition
outlet_faces = _bc.identify_faces(_fluid_mesh, edges=[geom.edges[5]], tol=face_tol)
n = len(outlet_faces)
v_vel = (_FluidField.V_VEL, jnp.zeros(n))
pres = (_FluidField.PRESSURE, jnp.zeros(n))
outlet_face_val = [v_vel, pres]


fluid_inlet_top_bc = _bc.DirichletBC(
  elem_faces=inlet_top_faces, values=inlet_top_face_val, name="inlet_top"
)
fluid_outlet_bc = _bc.DirichletBC(
  elem_faces=outlet_faces, values=outlet_face_val, name="outlet"
)

fluid_wall_bc = _bc.DirichletBC(
  elem_faces=wall_faces, values=wall_face_val, name="wall"
)

fluid_bcs_list = [
  fluid_wall_bc,
  fluid_inlet_top_bc,
  fluid_outlet_bc,
]

fluid_bc = _bc.process_boundary_conditions(
  fluid_bcs_list,
  _fluid_mesh,
)

_viz.plot_bc(fluid_bcs_list, _fluid_mesh)

# Solver

The incompressible Navier–Stokes–Brinkman system is **intrinsically non-linear** because of the convective term and the design-dependent viscosity .
Our solver therefore employs a *damped/modified Newton–Raphson* loop. For more details see 'solver.py'.

In [ ]:
solver_settings = {
  "linear": {
    "solver": _solv.LinearSolvers.PETSC,
    "petsc_solver": {},
  },
  "nonlinear": {"max_iter": 10, "threshold": 1.0e-8},
}
flow_solver = _fea.FluidSolver(
  mesh=_fluid_mesh,
  bc=fluid_bc,
  material=mat_params,
  solver_settings=solver_settings,
)


# Initialize

We initialize the FEA solver with the mesh, material, boundary conditions and solver settings. We provide an inital guess of the solution with the Dirichlet condition enforced.

In [ ]:
init_press_vel = jnp.zeros((_fluid_mesh.num_dofs,))
init_press_vel = init_press_vel.at[fluid_bc["fixed_dofs"]].set(fluid_bc["dirichlet_values"])

In [ ]:
u_velocity = init_press_vel[1 : _fluid_mesh.num_dofs : 3]
u_vel_elem = np.mean(u_velocity[_fluid_mesh.elem_nodes], axis=1)
_viz.plot_grid_mesh(_fluid_mesh, u_vel_elem)

# Concentration Transport

In [ ]:
concentration_mesh = _mesher.grid_mesh_brep(
  brep=geom,
  nelx_desired=nelx_desired,
  nely_desired=nely_desired,
  dofs_per_node=1,
  gauss_order=gauss_order,
)

In [ ]:
def initial_hill_concentration(x,y, x0=0.25, y0=0.5, sigma=0.03):
    """Initial concentration field.
    A Gaussian "hill" centered at (x0, y0) with standard deviation 0.007.
    Args:
        x: jnp.ndarray of shape (num_points, 1)
        y: jnp.ndarray of shape (num_points, 1)
        x0: float, x-coordinate of the center of the hill
        y0: float, y-coordinate of the center of the hill
        sigma: float, standard deviation of the Gaussian hill
    Returns:
        jnp.ndarray of shape (num_points, 1)
    """
    return jnp.exp(-(((x - x0) ** 2 + (y - y0) ** 2) / (sigma ** 2)))
    

In [ ]:
# outlet condition # wall condition are not required as Neumann BCs are natural
c_in = 1.0 # Inlet concentration value
# inlet top edge
inlet_top_faces = _bc.identify_faces(concentration_mesh, edges=[geom.edges[1]])
n = len(inlet_top_faces)
tv_top = (_ConcentrationField.CONCENTRATION, jnp.ones(n) * c_in) # c_in e.g
inlet_top_bc = _bc.DirichletBC(elem_faces=inlet_top_faces, values=[tv_top], name="inlet_top")

concentration_bc_list = [inlet_top_bc]
concentration_bc = _bc.process_boundary_conditions(concentration_bc_list, concentration_mesh)
init_concentration = jnp.zeros((concentration_mesh.num_dofs,))
init_concentration = init_concentration.at[concentration_bc["fixed_dofs"]].set(concentration_bc["dirichlet_values"])

In [ ]:
init_concentration_dof = init_concentration[concentration_mesh.elem_dof_mat]
elem_init_concentration = jnp.mean(init_concentration_dof, axis=1)
conc_ext = _Ext(min=jnp.min(init_concentration), max=jnp.max(init_concentration))

In [ ]:
_, ax = plt.subplots(1, 1)
ax = _viz.plot_grid_mesh(mesh=concentration_mesh, field=elem_init_concentration, ax=ax, colorbar=True)
ax.set_title("Initial Concentration Magnitude")

fig, ax = plt.subplots(figsize=(3, 1.5))
_viz.plot_bc(concentration_bc_list, concentration_mesh, ax=ax)
ax.set_title("Concentration Boundary Conditions")
plt.show()

In [ ]:
solver_settings = {
  "linear": {
    "solver": _solv.LinearSolvers.PETSC,
    "petsc_solver": {},
  },
  "nonlinear": {"max_iter": 10, "threshold": 1.0e-4},
}

concentration_solver = _fea_transient_transport.FEA(
  mesh=concentration_mesh,
  material=None,
  bc=concentration_bc,
  solver_settings=solver_settings,
)

# Time and time steps for the transient transport

In [ ]:
t_end = 0.9  # seconds
num_time_steps = 160
time_step_size = (t_end/num_time_steps) * jnp.ones(num_time_steps) #in seconds

time_step_sizes_star = time_step_size * char_velocity / char_length # non dimensionalize the time

# Specify time intervals in which the contrast is injected.

In [ ]:
start_times = jnp.array([0.0, 0.2, 0.4])
end_times  = jnp.array([0.1, 0.3, 0.5])

In [ ]:
concentration_transport_mat = _mat.SpeciesTransportMaterial(D=5.e-3,)  # Diffusion coefficient

eff_diffusivity_ext = _utils.Extent(min=1.e-6, max=concentration_transport_mat.D)

In [ ]:
characteristic_length = char_length * jnp.ones(concentration_mesh.num_elems) # length in y-direction

In [ ]:

min_inv_permeability = _mat.brinkman_bound(
  mat_params.dynamic_viscosity, 100.0 * concentration_mesh.bounding_box.lx
)
max_inv_permeability = _mat.brinkman_bound(
  mat_params.dynamic_viscosity, 5.0e-3 * concentration_mesh.bounding_box.lx
)
init_inv_permeability = _mat.brinkman_bound(
  mat_params.dynamic_viscosity, 1.0e-1 * concentration_mesh.bounding_box.lx
)   

inv_permeability_ext = _utils.Extent(min=min_inv_permeability, max=max_inv_permeability)

min_mat_frac = 0.67255178
init_ramp_penalty = _mat.calculate_initial_ramp_penalty(
  inv_permeability_ext, init_inv_permeability, min_mat_frac
)


# For optimizing the velcoity inlet via scaling

In [ ]:
bc_all = flow_solver.bc
bc_inlet_top = _bc.process_boundary_conditions([fluid_inlet_top_bc], _fluid_mesh)

fixed_np = np.asarray(bc_all["fixed_dofs"])
fixed_top_np = np.asarray(bc_inlet_top["fixed_dofs"])

pos_top = np.where(np.isin(fixed_np, fixed_top_np))[0]

pos_top = jnp.asarray(pos_top)
dirich_vals = jnp.asarray(bc_all["dirichlet_values"])
fixed = jnp.asarray(fixed_np)

# Optimize

In [ ]:
@functools.partial(jax.jit, static_argnames=("flow_solver",))
def objective_function(
  mat_frac: jnp.ndarray,
  flow_solver: _fea.FluidSolver,
  ramp_penalty: float,
  diffusion_ramp_penalty: float,
  thresh_beta: float,
  init_press_vel=None,
):
  """Computes the objective function and its gradient.

  This function computes the objective as the total dissipated power in the flow field,
    which is the sum of the dissipated power in each element. The dissipated power is
    computed using Brinkman penalty and the viscosity of the fluid. The Brinkman
    penalty is computed using a convex ramp function based on the material fraction.
    The viscosity of the fluid is computed using the shear rate in each element using the
    Non-Newtonian fluid model.
  Args:
    mat_frac: Material fraction array of shape (num_elems,). The array should contain
      values between 0 and 1, representing the material fraction at each element. Where
      0 indicates fluid and 1 indicates material. The material fraction can assume
      intermediate values between 0 and 1 during optimization.
    flow_solver: Fluid solver instance.
    ramp_penalty: Penalty factor for the Brinkman interpolation using a convex ramp
      function. The ramp penalty is  updated using a continuation scheme during
      optimization. In the beginning, the ramp penalty is set to a large value (which
      makes the mat frac vs Brinkman penalty convex which can be determined by
      brink_iter_factor) to allow fluid flow through the entire domain. As the
      optimization progresses, the ramp penalty is reduced to allow the material
      fraction to converge to material or fluid.
    diffusion_ramp_penalty: Penalty factor for the effective diffusion coefficient interpolation
      using a concave ramp function. The ramp penalty is updated using a continuation scheme during optimization.
    init_press_vel: Initial pressure-velocity field of shape (num_dofs,). This is used
      as the initial guess for the pressure-velocity field to Newton-Raphson
      iterations. It contains the Dirichlet boundary conditions applied to the
      pressure-velocity field.

  Returns:
    A tuple containing the objective value and a tuple of pressure-velocity field and
      material fraction.
  """

  def objective_wrapper(design_var):
    num_elems = flow_solver.mesh.num_elems
    mat_frac = design_var[:num_elems, ]     # (num_elems,)
    scale_frac = design_var[num_elems:, ]   # (number of inlets,)
    scale_var = scale_frac * jnp.asarray(a_ref)
    
    m = dens_filter @ mat_frac
    mat_frac_thresh = _utils.threshold_filter(m, beta=thresh_beta)
    brinkman_penalty = _mat.compute_ramp_interpolation(
      prop=mat_frac_thresh,
      ramp_penalty=ramp_penalty,
      prop_ext=inv_permeability_ext,
      mode="convex",
    )


    vals_scaled = dirich_vals
    vals_scaled = vals_scaled.at[pos_top].set(scale_var[0] * dirich_vals[pos_top])
    # vals_scaled = vals_scaled.at[pos_top].set(dirich_vals[pos_top])

    init_press_vel_s = init_press_vel.at[fixed].set(vals_scaled)

    press_vel = _solv.modified_newton_raphson_solve(
        flow_solver, init_press_vel_s, brinkman_penalty
    )

    press_vel = press_vel.at[fixed].set(vals_scaled)

    obj_args = (
      brinkman_penalty,
      press_vel[_fluid_mesh.elem_dof_mat],
      _fluid_mesh.elem_node_coords,
    )

    press_vel_elem = press_vel[_fluid_mesh.elem_dof_mat]
    elem_velocity = jnp.stack([press_vel_elem[:, 1::3], press_vel_elem[:, 2::3]], axis=2)  # shape (num_elems, num_nodes, 2)
    elem_velocity = elem_velocity.reshape(elem_velocity.shape[0], -1)  # shape (num_elems, 2*num_nodes)
    elem_velocity_star = elem_velocity / char_velocity

    eff_diffusivity_ramp = _mat.compute_ramp_interpolation(
      prop=(1 - mat_frac_thresh.reshape(-1,)),
      ramp_penalty=diffusion_ramp_penalty,
      prop_ext=eff_diffusivity_ext,
      mode="convex"
    )
    # solve the FE problem
    conc_t_hist = _fea_transient_transport.solve_transient_fea(
        concentration_solver,
        init_concentration,
        eff_diffusivity_ramp,
        time_step_sizes_star,
        characteristic_length,
        elem_velocity_star,
        char_velocity=char_velocity,
        dt_phys=time_step_size,     # NEW
        intervals_start=start_times,
        intervals_end=end_times,
    )
    # dimensionalize the concentration
    conc_t_hist_d = conc_t_hist * conc_ext.range + conc_ext.min # Shape: (num_time_steps, n_dofs) 

    def loss_fn(conc_t: jnp.ndarray):
      """
        conc_t: dimensionalized concentration at all time steps, shape (num_time_steps, n_dofs)
      """
      n_det = sngrm_grndtruth.shape[1]
      n_theta = sngrm_grndtruth.shape[2]
      sinograms_pred = _radon.sinogram_from_conc(conc_t, nely_desired+1, nelx_desired+1, n_theta= n_theta, n_det=n_det)

      theta_idx = jnp.arange(0, n_theta, 2)   # 2 = keep every other projection angle
      
      diff = sinograms_pred[:, :, theta_idx] - sngrm_grndtruth[:, :, theta_idx]
      obj = jnp.mean(diff**2)

      return obj, sinograms_pred


    obj, sinograms_pred = loss_fn(conc_t_hist_d)
    return obj, (press_vel, mat_frac_thresh, sinograms_pred)

  (obj, (press_vel, mat_frac_thresh, sinograms_pred)), d_obj = jax.value_and_grad(
    objective_wrapper, has_aux=True
  )(mat_frac)
  return obj, d_obj.reshape((-1, 1)), press_vel, mat_frac_thresh, sinograms_pred

# For optimizing velocity inlet scaling

In [ ]:
num_inlets = 1
a_ref = np.array([1.0])

# Physical admissible range
a_min = np.array([0.0])
a_max = np.array([2.0])

# Physical initial guess
a0 = np.array([0.1])

x0 = a0 / a_ref
LB = a_min / a_ref
UB = a_max / a_ref

# Optimize
Here, we define the optimization loop. We use the MMA optimizer. We begin the optimization with a uniform design corresponding to the minimum allowed material/solid fraction. 

In [ ]:
def plot_hist_mat_frac(mat_frac_thresh, bins=50):
    m = np.array(jax.device_get(mat_frac_thresh)).ravel()
    plt.figure()
    plt.hist(m, bins=bins)
    plt.xlabel("mat_frac_thresh value")
    plt.ylabel("count")
    plt.title("Histogram of mat_frac_thresh")
    plt.show()

In [ ]:
def optimize_design(
  fe: _fea.FluidSolver,
  min_mat_frac: float,
  max_iter: int,
  move_limit: float = 1e-2,
  kkt_tol: float = 1e-3,
  step_tol: float = 1e-3,
  plot_interval: int = 5,
):
  """Optimizes the design using the Method of Moving Asymptotes (MMA) optimization.

  The function initializes the design variables with the minimum material fraction and
    then iteratively updates them via the MMA approach. At each iteration, the objective
    function and its gradient are computed using JAX `value_and_grad` function based
    on the current material distribution. Additionally, a material constraint is enforced
    to ensure the design maintains at least the specified minimum material fraction.

  Args:
    fe: Fluid solver instance containing mesh, boundary conditions, and material
      properties.
    min_mat_frac: Minimum allowed material fraction to enforce material presence in the
      design.
    max_iter: Maximum number of iterations for the MMA optimization loop.
    move_limit: Maximum allowable change in the design variable per iteration.
    kkt_tol: Tolerance for the Karush-Kuhn-Tucker (KKT) optimality criteria.
    step_tol: Tolerance for the optimization step size.
    plot_interval: Number of iterations between plot updates.

  Returns:
    mma_state: The final state of the MMA optimization, including the optimized design
      variables, iteration count, and convergence flag.
    history: A dictionary recording the history of the objective function values and
      constraint violations with keys:
      'obj'      - list of objective function values,
      'vol_cons' - list of volume constraint values.
  """

  gamma0 = 0.5 * np.ones((fe.mesh.num_elems, 1))
  x_a0 = (a0 / a_ref).reshape(num_inlets, 1)

  design_var = np.vstack([gamma0, x_a0])
  num_design_var = design_var.shape[0]
  num_cons = 1
  lower_bound = np.zeros((num_design_var, 1))
  upper_bound = np.ones((num_design_var, 1))
  lower_bound[-num_inlets:, 0] = LB
  upper_bound[-num_inlets:, 0] = UB
  history = {"obj": [], "vol_cons": []}

  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=kkt_tol,
    step_tol=step_tol,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=num_cons,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
  )
  mma_state = _mma.init_mma(design_var, mma_params)

  press_vel = jnp.zeros((fe.mesh.num_dofs,))
  press_vel = press_vel.at[fe.bc["fixed_dofs"]].set(fe.bc["dirichlet_values"])


  while not mma_state.is_converged:
    fluid_ramp_penalty = max(30.0, 75.0 - 0.30 * mma_state.epoch)
    thresh_beta = min(6.0, 1.0 + 0.03 * mma_state.epoch)
    diffusion_ramp_penalty = min(10.0, 5.0 + 0.05 * mma_state.epoch)

    objective, grad_obj, press_vel, mat_frac, sinogram_pred = objective_function(
      mma_state.x, fe, fluid_ramp_penalty, diffusion_ramp_penalty, thresh_beta, press_vel
    )
    constr = np.array([[-1.0]], dtype=float)     # (m,1) = (1,1)
    grad_cons = np.zeros((1, num_design_var), dtype=float)    # (1, num_design_var)
    press_vel = jax.lax.stop_gradient(press_vel)
    press_vel = press_vel.at[fe.bc["fixed_dofs"]].set(fe.bc["dirichlet_values"])

    mma_state = _mma.update_mma(
      mma_state, mma_params, objective, grad_obj, constr, grad_cons
    )

    status = f"epoch {mma_state.epoch} J {objective:.2E}"
    print(status)

    
    if mma_state.epoch % plot_interval == 0 or mma_state.epoch == 1:
      _, ax = plt.subplots(1, 1)
      ax = _viz.plot_grid_mesh(
        mesh=fe.mesh,
        field=np.round(mat_frac).reshape(-1),
        ax=ax,
        colorbar=False,
        cmap=_cmap,
        val_range=(0.0, 1.0),
      )
      ax.set_xticks([])
      ax.set_yticks([])
      for spine in ax.spines.values():
        spine.set_visible(False)

      press_vel_elem = press_vel[fe.mesh.elem_dof_mat]
      press_elem = np.mean(press_vel_elem[:, 0::3], axis=1)
      u_vel_elem = np.mean(press_vel_elem[:, 1::3], axis=1)
      v_vel_elem = np.mean(press_vel_elem[:, 2::3], axis=1)
      vel_elem_mag = np.sqrt(u_vel_elem**2 + v_vel_elem**2)
      _, ax = plt.subplots(1, 1)
      ax = _viz.plot_grid_mesh(mesh=fe.mesh, field=vel_elem_mag, ax=ax, colorbar=True)
      ax.set_title("Velocity Magnitude")

      _, ax = plt.subplots(1, 1)
      ax = _viz.plot_grid_mesh(mesh=fe.mesh, field=press_elem, ax=ax, colorbar=True)
      ax.set_title("Pressure")
      plt.show()
      # plot_hist_mat_frac(mat_frac)

  return mma_state, history

In [ ]:
mma_state, u = optimize_design(
  flow_solver, min_mat_frac=min_mat_frac, max_iter=300, kkt_tol=1.e-3, step_tol = 1.e-3, move_limit=3.e-2, plot_interval=10
)

In [ ]:
np.save("stenosis_state_x_vel_op.npy", mma_state.x)

In [ ]:
# compute MSE between saved MMA states
a = jnp.load("stenosis_state_x_vel_op.npy")
b = jnp.load("mma_state_x_stenosis.npy")

d_var = jnp.asarray(a).reshape(-1)
b = jnp.asarray(b).reshape(-1)


In [ ]:
mat_frac_op = d_var[:_fluid_mesh.num_elems]
dens_filter_op = _utils.create_density_filter(
  _fluid_mesh.elem_centers,
  cutoff_distance=0.0365 * _fluid_mesh.bounding_box.diag_length,
  filter_type=_utils.Filters.CIRCULAR,
)
m_output = dens_filter_op @ mat_frac_op
mat_frac_out = _utils.threshold_filter(m_output, beta=4.)

# display-only smoothing
from scipy.ndimage import gaussian_filter
mat_frac_plot = gaussian_filter(

    np.array(mat_frac_out).reshape(nely_desired, nelx_desired),

    sigma=2.0

).reshape(-1)

In [ ]:
a_scale = d_var[_fluid_mesh.num_elems:]
m_input = dens_filter_op @ b
mat_frac_in = _utils.threshold_filter(m_input, beta=4.)

In [ ]:
gamma_out = np.asarray(mat_frac_out).reshape(-1)
gamma_true = np.asarray(mat_frac_in).reshape(-1)

# -----------------------------
# 1. nRMSE_gamma / relative L2 error
# -----------------------------
numerator = np.sqrt(np.sum((gamma_out - gamma_true) ** 2))
denominator = np.sqrt(np.sum(gamma_true ** 2)) + 1e-12

nrmse_gamma = numerator / denominator
nrmse_gamma_percent = 100.0 * nrmse_gamma

# -----------------------------
# 2. Dice coefficient for thresholded fluid mask
#    fluid mask: gamma <= threshold
# -----------------------------
thr = 0.5

fluid_out = gamma_out <= thr
fluid_true = gamma_true <= thr

tp = np.sum(fluid_out & fluid_true)
fp = np.sum(fluid_out & ~fluid_true)
fn = np.sum(~fluid_out & fluid_true)

dice = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-12)

# -----------------------------
# 3. Volume fraction difference
# -----------------------------
volfrac_out = np.mean(gamma_out)
volfrac_true = np.mean(gamma_true)
delta_volfrac = volfrac_out - volfrac_true

# -----------------------------
# Concise report
# -----------------------------
print(f"nRMSE_gamma / relative L2 error : {nrmse_gamma:.6f}")
print(f"nRMSE_gamma (%)                 : {nrmse_gamma_percent:.2f}%")
print(f"Dice coefficient                : {dice:.6f}")
print(f"Volume fraction output          : {volfrac_out:.6f}")
print(f"Volume fraction true            : {volfrac_true:.6f}")
print(f"Δ volume fraction               : {delta_volfrac:+.6f}")

In [ ]:
fig, ax = plt.subplots(1, 1)

ax = _viz.plot_grid_contour(
    mesh=concentration_mesh,
    field=np.array(mat_frac_out).reshape(-1),
    ax=ax,
    cmap=_viz.fluid_cmap,
    contour_level=0.5,
    binary_fill=True,
    colorbar=False,
)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.axis("off")

fig.savefig("outputs/stenosis_op.svg", format="svg", bbox_inches="tight", pad_inches=0)
plt.show()